## Installation

You can install Meterstick either from pip for the stable version
```
!pip install meterstick
```
or from GitHub for the latest version.
```
!git clone https://github.com/google/meterstick.git
import sys, os
sys.path.append(os.getcwd())
```

In [ ]:
import pandas as pd

try:  # try to read from local file first
  df = pd.read_csv('datasets/baseball_pitches_to_SEA_batters.csv')
  player_names = pd.read_csv('datasets/baseball_player_names.csv')
except:  # fall back to read from web
  print('Reading from Kaggle...')
  import kagglehub

  # Download latest version
  path = kagglehub.dataset_download('pschale/mlb-pitch-data-20152018')
  df_pitches = pd.read_csv(f'{path}/pitches.csv')
  df_atbats = pd.read_csv(f'{path}/atbats.csv')
  player_names = pd.read_csv(f'{path}/player_names.csv')
  # Get IDs of opening-day roster of Mariners in 2019 (https://www.baseball-reference.com/boxes/OAK/OAK201903200.shtml)
  sea_batters_names = [
      ('Dee', 'Gordon'),
      ('Mitch', 'Haniger'),
      ('Jay', 'Bruce'),
      ('Edwin', 'Encarnacion'),
      ('Domingo', 'Santana'),
      ('Omar', 'Narvaez'),
      ('Ryon', 'Healy'),
      ('Tim', 'Beckham'),
      ('Ichiro', 'Suzuki'),
  ]

  # Look up IDs of these players
  sea_batters_ids = []
  for first_name, last_name in sea_batters_names:
    player_row = player_names[
        (player_names['first_name'] == first_name)
        & (player_names['last_name'] == last_name)
    ]
    if player_row.empty:
      print(f'Player {first_name} {last_name} not found')
    else:
      sea_batters_ids.append(player_row.iloc[0]['id'])
  df = df_atbats[df_atbats['batter_id'].isin(sea_batters_ids)].merge(
      df_pitches, on='ab_id', how='inner'
  )[['ab_id', 'pitch_type', 'pitcher_id', 'batter_id']]

player_names['Name'] = (
    player_names['first_name'] + ' ' + player_names['last_name']
)
df = df.merge(
    player_names[['id', 'Name']],
    how='left',
    left_on='pitcher_id',
    right_on='id',
)
df = df.merge(
    player_names[['id', 'Name']],
    how='left',
    left_on='batter_id',
    right_on='id',
    suffixes=('_pitcher', '_batter'),
)
df

,ab_id,pitch_type,pitcher_id,batter_id
0,2015000146,FF,527054,543829
1,2015000146,FF,527054,543829
2,2015000161,FF,527054,543829
3,2015000161,SL,527054,543829
4,2015000161,FF,527054,543829
...,...,...,...,...
55393,2018185558,FF,458677,570267
55394,2018185558,FF,458677,570267
55395,2018185558,FF,458677,570267
55396,2018185558,FF,458677,570267


In [ ]:
from meterstick import *

pitch_types = Count("pitch_type") | Distribution(over="pitch_type")
pitch_types.compute_on(df, split_by=["Name_batter", "Name_pitcher"])

Distribution of count(pitch_type)
Name_batter Name_pitcher pitch_type                                   
Dee Gordon  A.J. Cole    CH                                   0.078431
                         CU                                   0.196078
                         FF                                   0.431373
                         FT                                   0.137255
                         SL                                   0.156863
...                                                                ...
Tim Beckham Zack Greinke FT                                   0.200000
                         SL                                   0.600000
            Zack Wheeler FF                                   0.375000
                         FT                                   0.375000
                         SL                                   0.250000

[11436 rows x 1 columns]